In [122]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from pathlib import Path
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ultralytics import YOLO
from src.utils_MF import *
from src.utils_MFP import *
from src.utils_LF import *

In [ ]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["ULTRALYTICS_IGNORE_MULTIPROCESS"] = "1"

In [112]:
ROOT = Path().resolve()

model_rgb = YOLO("runs/detect/wildlife_rgb_v8n_cpu5/weights/best.pt")
model_t   = YOLO("runs/detect/wildlife_t_v8n_cpu/weights/best.pt")

print("Modelos cargados:")
print("  RGB -> runs/detect/wildlife_rgb_v8n_cpu5/weights/best.pt")
print("  T   -> runs/detect/wildlife_t_v8n_cpu/weights/best.pt")

IMG_ROOT_RGB = ROOT / "data" / "format_rgb" / "images"
IMG_ROOT_T   = ROOT / "data" / "format_t"   / "images"
LBL_ROOT     = ROOT / "data" / "format_rgb" / "labels"

TRAIN_RGB_DIR = IMG_ROOT_RGB / "train"
VAL_RGB_DIR   = IMG_ROOT_RGB / "val"
TEST_RGB_DIR  = IMG_ROOT_RGB / "test"

TRAIN_T_DIR   = IMG_ROOT_T / "train"
VAL_T_DIR     = IMG_ROOT_T / "val"
TEST_T_DIR    = IMG_ROOT_T / "test"

GT_TRAIN_DIR = LBL_ROOT / "train"
GT_VAL_DIR   = LBL_ROOT / "val"
GT_TEST_DIR  = LBL_ROOT / "test"

NUM_CLASSES = 3
CLASS_LABELS = ["Cow", "Deer", "Horse"]
class_names = {0: "Cow", 1: "Deer", 2: "Horse"}

IMG_SIZE = 640


Modelos cargados:
  RGB -> runs/detect/wildlife_rgb_v8n_cpu5/weights/best.pt
  T   -> runs/detect/wildlife_t_v8n_cpu/weights/best.pt


In [ ]:
LF_ROOT = ROOT / "runs" / "late_fusion"
OUT_LF_TRAIN_IMG_DIR  = LF_ROOT / "train"
OUT_LF_TRAIN_PRED_DIR = LF_ROOT / "preds_train"
OUT_LF_VAL_IMG_DIR    = LF_ROOT / "val"
OUT_LF_VAL_PRED_DIR   = LF_ROOT / "preds_val"
OUT_LF_TEST_IMG_DIR   = LF_ROOT / "test"
OUT_LF_TEST_PRED_DIR  = LF_ROOT / "preds_test"

for d in [
    OUT_LF_TRAIN_IMG_DIR, OUT_LF_TRAIN_PRED_DIR,
    OUT_LF_VAL_IMG_DIR,   OUT_LF_VAL_PRED_DIR,
    OUT_LF_TEST_IMG_DIR,  OUT_LF_TEST_PRED_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

run_middle_fusion_split(
    model_rgb=model_rgb,
    model_t=model_t,
    class_names=class_names,
    rgb_dir=TRAIN_RGB_DIR,
    t_dir=TRAIN_T_DIR,
    out_img_dir=OUT_LF_TRAIN_IMG_DIR,
    out_pred_dir=OUT_LF_TRAIN_PRED_DIR,
    img_size=IMG_SIZE,
)

run_middle_fusion_split(
    model_rgb=model_rgb,
    model_t=model_t,
    class_names=class_names,
    rgb_dir=VAL_RGB_DIR,
    t_dir=VAL_T_DIR,
    out_img_dir=OUT_LF_VAL_IMG_DIR,
    out_pred_dir=OUT_LF_VAL_PRED_DIR,
    img_size=IMG_SIZE,
)

# TEST
run_middle_fusion_split(
    model_rgb=model_rgb,
    model_t=model_t,
    class_names=class_names,
    rgb_dir=TEST_RGB_DIR,
    t_dir=TEST_T_DIR,
    out_img_dir=OUT_LF_TEST_IMG_DIR,
    out_pred_dir=OUT_LF_TEST_PRED_DIR,
    img_size=IMG_SIZE,
)

Encontradas 228 imágenes RGB en C:\Users\isiva\OneDrive\Documents\GitHub\TP-Final-Vision\data\format_rgb\images\train.
[OK] Middle Fusion: 020221_deer_pens_xt2_DJI_0306.JPG -> img:020221_deer_pens_xt2_DJI_0306.JPG, preds:020221_deer_pens_xt2_DJI_0306.txt
[OK] Middle Fusion: 020221_deer_pens_xt2_DJI_0306.JPG -> img:020221_deer_pens_xt2_DJI_0306.JPG, preds:020221_deer_pens_xt2_DJI_0306.txt
[OK] Middle Fusion: 020221_deer_pens_xt2_DJI_0392.JPG -> img:020221_deer_pens_xt2_DJI_0392.JPG, preds:020221_deer_pens_xt2_DJI_0392.txt
[OK] Middle Fusion: 020221_deer_pens_xt2_DJI_0392.JPG -> img:020221_deer_pens_xt2_DJI_0392.JPG, preds:020221_deer_pens_xt2_DJI_0392.txt
[OK] Middle Fusion: 022521_DJI_0024.JPG -> img:022521_DJI_0024.JPG, preds:022521_DJI_0024.txt
[OK] Middle Fusion: 022521_DJI_0024.JPG -> img:022521_DJI_0024.JPG, preds:022521_DJI_0024.txt
[OK] Middle Fusion: 022521_DJI_0034.JPG -> img:022521_DJI_0034.JPG, preds:022521_DJI_0034.txt
[OK] Middle Fusion: 022521_DJI_0034.JPG -> img:022521_D

In [ ]:
lf_dir = OUT_LF_VAL_PRED_DIR 

empty = 0
total = 0
for p in Path(lf_dir).glob("*.txt"):
    total += 1
    if p.read_text().strip() == "":
        empty += 1

print(f"Total txt LF: {total}, vacíos: {empty}")


Total txt LF: 32, vacíos: 2


In [ ]:
patch_dataset = PatchFusionDataset(
    rgb_dir=TRAIN_RGB_DIR,
    t_dir=TRAIN_T_DIR,
    preds_dir=OUT_LF_TRAIN_PRED_DIR,
    gt_dir=GT_TRAIN_DIR,
    img_size=IMG_SIZE,
    patch_size=64,
    iou_pos_th=0.5,
    iou_neg_th=0.3,
    max_samples_per_img=50,
)
len(patch_dataset)

patch_loader = DataLoader(
    patch_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

PatchFusionDataset: 414 samples


In [ ]:
patch_model = PatchFusionNet(in_channels=4).to("cpu")
optimizer   = torch.optim.Adam(patch_model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion   = nn.BCELoss()

EPOCHS = 10

for epoch in range(EPOCHS):
    patch_model.train()
    running_loss = 0.0

    for patches, labels in patch_loader:
        patches = patches.to("cpu")
        labels  = labels.to("cpu")

        optimizer.zero_grad()
        scores = patch_model(patches)

        loss = criterion(scores, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss / len(patch_loader):.4f}")

Epoch 1/10 - Loss: 0.4995
Epoch 2/10 - Loss: 0.3781
Epoch 3/10 - Loss: 0.3207
Epoch 4/10 - Loss: 0.3071
Epoch 5/10 - Loss: 0.2679
Epoch 6/10 - Loss: 0.2448
Epoch 7/10 - Loss: 0.2305
Epoch 8/10 - Loss: 0.1977
Epoch 9/10 - Loss: 0.1984
Epoch 10/10 - Loss: 0.1793


In [ ]:
PF_ROOT = ROOT / "runs" / "patch_fusion"
PF_ROOT.mkdir(parents=True, exist_ok=True)
PFN_PATH = PF_ROOT / "patch_fusion_net.pt"

torch.save(patch_model.state_dict(), PFN_PATH)
print("PatchFusionNet guardado en", PFN_PATH)

PatchFusionNet guardado en C:\Users\isiva\OneDrive\Documents\GitHub\TP-Final-Vision\runs\patch_fusion\patch_fusion_net.pt


In [ ]:
patch_model = PatchFusionNet(in_channels=4).to("cpu")
patch_model.load_state_dict(torch.load(PFN_PATH, map_location="cpu"))

OUT_PF_VAL_PRED_DIR  = PF_ROOT / "preds_val"
OUT_PF_TEST_PRED_DIR = PF_ROOT / "preds_test"

refine_with_patch_fusion(
    patch_model=patch_model,
    rgb_dir=VAL_RGB_DIR,
    t_dir=VAL_T_DIR,
    preds_late_dir=OUT_LF_VAL_PRED_DIR,
    preds_pf_dir=OUT_PF_VAL_PRED_DIR,
    img_size=IMG_SIZE,
    patch_size=64,
    score_thr=0.5,
    device="cpu",
)

refine_with_patch_fusion(
    patch_model=patch_model,
    rgb_dir=TEST_RGB_DIR,
    t_dir=TEST_T_DIR,
    preds_late_dir=OUT_LF_TEST_PRED_DIR,
    preds_pf_dir=OUT_PF_TEST_PRED_DIR,
    img_size=IMG_SIZE,
    patch_size=64,
    score_thr=0.5,
    device="cpu",
)

[PatchFusion] Refinando 32 imágenes...
[PatchFusion] Listo. Predicciones refinadas en C:\Users\isiva\OneDrive\Documents\GitHub\TP-Final-Vision\runs\patch_fusion\preds_val
[PatchFusion] Refinando 18 imágenes...
[PatchFusion] Listo. Predicciones refinadas en C:\Users\isiva\OneDrive\Documents\GitHub\TP-Final-Vision\runs\patch_fusion\preds_test


In [ ]:
# Late Fusion
metrics_lf_val = evaluate_yolo_predictions(
    pred_dir=OUT_LF_VAL_PRED_DIR,
    gt_dir=GT_VAL_DIR,
    num_classes=NUM_CLASSES,
    iou_threshold=0.5,
)
metrics_lf_test = evaluate_yolo_predictions(
    pred_dir=OUT_LF_TEST_PRED_DIR,
    gt_dir=GT_TEST_DIR,
    num_classes=NUM_CLASSES,
    iou_threshold=0.5,
)
print_metrics("Late Fusion (VAL)",  metrics_lf_val)
print_metrics("Late Fusion (TEST)", metrics_lf_test)

# Patch Middle Fusion
metrics_pf_val = evaluate_yolo_predictions(
    pred_dir=OUT_PF_VAL_PRED_DIR,
    gt_dir=GT_VAL_DIR,
    num_classes=NUM_CLASSES,
    iou_threshold=0.5,
)
metrics_pf_test = evaluate_yolo_predictions(
    pred_dir=OUT_PF_TEST_PRED_DIR,
    gt_dir=GT_TEST_DIR,
    num_classes=NUM_CLASSES,
    iou_threshold=0.5,
)
print_metrics("Patch Middle Fusion (VAL)",  metrics_pf_val)
print_metrics("Patch Middle Fusion (TEST)", metrics_pf_test)


Late Fusion (VAL) @ IoU 0.5
-----------------------------
mAP:       0.9118811260118532
Precision: 0.6870229007581143
Recall:    0.9574468085004526
F1:        0.7999999995064099
AP por clase: [np.float64(0.920677022915462), np.float64(0.867969386655121), np.float64(0.9469969684649766)]
Late Fusion (TEST) @ IoU 0.5
------------------------------
mAP:       0.987713647296882
Precision: 0.7460317460199043
Recall:    0.9999999999787235
F1:        0.854545454040496
AP por clase: [np.float64(0.9631409427391309), np.float64(0.9999999993333333), np.float64(0.9999999998181818)]
Patch Middle Fusion (VAL) @ IoU 0.5
-------------------------------------
mAP:       0.7438008574703562
Precision: 0.8021978021889868
Recall:    0.7765957446725894
F1:        0.7891891886807889
AP por clase: [np.float64(0.7303769105858005), np.float64(0.682573280940684), np.float64(0.818452380884584)]
Patch Middle Fusion (TEST) @ IoU 0.5
--------------------------------------
mAP:       0.8725231335779481
Precision: 0.84